In [1]:
!pip install fastapi uvicorn pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.0 MB/s eta 0:00:00


In [2]:
import os
import sklearn
import pandas as pd
from joblib import load
import json

In [3]:
!ngrok authtoken 2tQR3hMKgY1918rkKxTdGfu9g5s_4S59sDie9wVDY55tEjvZa

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [4]:
model = load('heart_lr_model.pkl')

In [5]:
model.feature_names_in_

array(['sbp', 'tobacco', 'ldl', 'adiposity', 'famhist', 'typea',
       'obesity', 'alcohol', 'age'], dtype=object)

In [ ]:
%%writefile app.py

import os
import sklearn
import pandas as pd
import numpy as np
from joblib import load
from fastapi import FastAPI, HTTPException
from contextlib import asynccontextmanager
from pydantic import BaseModel

app = FastAPI()

# Define input data schema
class PredictionRequest(BaseModel):
    sbp: int
    tobacco: float
    ldl: float
    adiposity: float
    famhist: str
    typea: int
    obesity: float
    alcohol: float
    age: int

ml_model = load('heart.pkl')

# Prediction endpoint
@app.post("/predict")
def predict(input_data: PredictionRequest):
    try:
        # Convert input data to a dictionary for prediction
        input_dict = input_data.dict()

        df = pd.DataFrame(input_dict, index = [0])

        # Call the model's prediction method
        prediction = ml_model.predict(df)

        # Return the prediction result
        return {f"Patient has CHD (0/1) {np.round(prediction[0], 1)}"}

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction error: {e}")


Writing app.py


In [ ]:
!nohup uvicorn app:app --host 0.0.0.0 --port 6010 &

nohup: appending output to 'nohup.out'


In [ ]:
!ps -ax | grep uvicorn

   1723 ?        Rl     0:01 /usr/bin/python3 /usr/local/bin/uvicorn app:app --host 0.0.0.0 --port 6
   1733 ?        S      0:00 /bin/bash -c ps -ax | grep uvicorn
   1735 ?        S      0:00 grep uvicorn


In [ ]:
from pyngrok import ngrok

# Expose the FastAPI app
public_url = ngrok.connect(6010)
print(f"Public URL: {public_url}")

Public URL: NgrokTunnel: "https://3bc6-34-106-205-48.ngrok-free.app" -> "http://localhost:6010"


## Alert!

Run the following commands only at the end, to stop the ngrok and uvicorn service


In [ ]:
ngrok.kill()

In [ ]:
!kill -9 <pid of uvicorn service>

/bin/bash: -c: line 1: syntax error near unexpected token `newline'
/bin/bash: -c: line 1: `kill -9 <pid of uvicorn service>'
